# Member 2: Pretrained YOLO + ByteTrack experiments
SBES 426 — Medical Pattern Recognition | Phase 2 | Validation match **117093**.

**Provisional local scientific validation. Official E3/E4, AP, HOTA, IDF1 and MOTA are blocked by unimplemented shared dependencies. Held-out testing and fine-tuning were not run.**

This notebook loads verified caches by default. It does not train, download or rerun detection. Detection estimates boxes and confidence scores; tracking associates observations across time. Neither directly estimates physiological fatigue.

In [ ]:
from pathlib import Path
import json, hashlib, os
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/yolo.yaml').is_file())
RUN = ROOT / "outputs/member2_scientific/runs/member2_validation_20261125"
manifest = json.loads((RUN/'manifest.json').read_text())
def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()
hashes = json.loads((RUN/'artifact_hashes.json').read_text())
assert all(sha(RUN/p)==value for p,value in hashes.items()), 'Saved artifact changed'
assert manifest['split']=='val' and manifest['match']=='117093'
weights = Path('F:/Football-Fatigue-Project/models/yolov8n.pt')
assert weights.is_file() and sha(weights)==manifest['weights_sha256']
def show(relative, width=1000): display(Image(filename=str(RUN/relative), width=width))
display(pd.DataFrame([{'run_id':manifest['run_id'], 'status':manifest['status'], 'verified_artifacts':len(hashes), 'device':manifest['device']}]))

## 1. Problem definition and objectives
Detect the annotated football players as full-frame top-left `(x,y,w,h)` boxes, then attach temporary positive IDs to their observations. Confidence measures the detector's person score, not identity certainty. Ground truth supplies annotated player positions and original player IDs for evaluation, never for suppressing detections at inference.

We test zero-shot transfer from COCO people to tiny panoramic players. Pipeline B is independent of Member 1's proposal-dependent HOG/SVM and small-CNN detectors.

In [ ]:
display(pd.DataFrame([{'detector':'yolov8n_coco','tracker':'bytetrack','class':'COCO person (0)','training_this_run':'none','target':'22 annotated players/frame'}]))

## 2. Dataset description and ground truth
SoccerTrack v2 release v1.2, revision `325bd4c57ee8e135ab652f3921d274526ada496b`. Each validation frame has 22 annotated player boxes. Referees, staff and spectators can be genuine people without player annotations. A person detection may therefore be counted as a false positive by the player benchmark.

Raw MOT is already 0-based. Sequential `data_io.iter_frames` avoids the clip's container-time offset. We reuse Member 1's raw parser/geometry cleaner; official GT conversion remains gated by Member 3 validation.

In [ ]:
display(pd.DataFrame([manifest['video_info']]))
display(pd.DataFrame([manifest['gt_cleaning_issues']]))
show('figures/gt_frame_0.jpg')

## 3. Data preparation and approved split
Train: 118575, 118576, 118577, 118578, 128058. Validation: 117093. Optional 117092 stays disabled. Held-out 128057 and 132831 were not accessed.

There is no new YOLO training, normalization fitted to GT, class resampling or augmentation in this inference workflow. Ultralytics letterboxes each tile and normalizes model input internally. Controlled experiments use identical development frames, disjoint from the required reporting frames. The entire validation match remains development data at the match level.

In [ ]:
development = manifest['development_frame_ids']
report = manifest['required_union_frame_ids']
assert not set(development) & set(report)
display(pd.DataFrame([{'development_frames':len(development),'required_union_frames':len(report),'overlap':len(set(development)&set(report))}]))
print('Development frame IDs:', development)

## 4. Pretrained model and weight provenance
Only the existing absolute local `yolov8n.pt` is loaded, after SHA-256 verification. No models, datasets or dependencies are downloaded. `YOLO_OFFLINE=true` and `YOLO_AUTOINSTALL=false` suppress automatic network checks/installations. The model is COCO-pretrained; this team did not train it in Phase 2. A trusted local serialized checkpoint is required; do not load arbitrary uploaded checkpoints.

In [ ]:
display(pd.DataFrame([{'weights_sha256':manifest['weights_sha256'],'git_commit':manifest['git_commit'],'dataset_revision':manifest['dataset_revision'],'created_utc':manifest['created_utc']}]))
display(pd.DataFrame(json.loads((RUN/'environment.json').read_text())['packages'].items(),columns=['package','version']))

## 5. YOLOv8n architecture and training mechanism
The loaded network has Conv/C2f backbone blocks and SPPF context pooling. Upsampling, concatenation and C2f blocks aggregate features across scales; the Detect head predicts at strides 8, 16 and 32. It is anchor-free with respect to predefined anchor-box shapes: predictions use grid anchor points and learned box-distance distributions.

The installed `v8DetectionLoss` uses BCEWithLogits classification, CIoU box loss and Distribution Focal Loss for distance-bin regression; TaskAlignedAssigner assigns training targets. These describe the pretrained architecture's training mechanism, **not losses measured in this experiment**. No optimizer, epochs, training curve or new loss values are claimed. Native small-object information can still be lost through feature downsampling.

In [ ]:
architecture = json.loads((RUN/'architecture.json').read_text())
display(pd.DataFrame([{k:architecture[k] for k in ['parameters','head','nc','reg_max','strides']}]))
display(pd.DataFrame(architecture['layers']))

## 6. Small-object detection challenges
A 20-pixel-wide player shrinks to about 6.25 pixels when a 4096-pixel panorama is reduced to 1280 pixels. A 1280-wide native tile avoids that horizontal shrink at `imgsz=1280`. Smaller tiles add inference calls and can truncate players; overlap helps preserve border context. Tiling cannot recover genuinely missing visual detail, heavy occlusion or annotation mismatch.

In [ ]:
display(pd.DataFrame([{'source_player_width':20,'whole_frame_width':4096,'input_width':1280,'resized_player_width':20*1280/4096}]))
show('required_validation/stages/original.jpg')

## 7. Tiling and tile coverage
Tiles cover every native pixel, with the final tile aligned to the image edge. The configured overlap is a minimum; edge alignment may create larger overlap. Synthetic coverage tests include 4096×1080, 3840×1504, exact tile sizes and frames smaller than a tile. Invalid dimensions and overlap are rejected.

In [ ]:
geometry = json.loads((RUN/'required_validation/stages/geometry.json').read_text())
display(pd.DataFrame(geometry['tiles'],columns=['x0','y0','x1','y1']))
show('required_validation/stages/tiles.jpg')
show('figures/overlap_crop.jpg',width=560)

## 8. Tile-local YOLO detection
BGR uint8 crops are passed together to the installed `YOLO.predict`, requesting only class 0. Per-tile NMS IoU is 0.5 and confidence floor is 0.05. Predictions are already expressed in each crop's native pixel coordinates after Ultralytics reverses its letterbox transform. Keeping the low floor supports both reporting-threshold curves and ByteTrack's low-score association stage.

In [ ]:
display(pd.DataFrame([manifest['selected_configuration']['inference']]))
show('required_validation/stages/tile_local.jpg',width=850)

## 9. Global coordinates and non-maximum suppression
For tile origin `(x0,y0)`, add `[x0,y0,x0,y0]` to local xyxy, clip to the full image, then perform torchvision global NMS at IoU 0.5. Convert retained boxes to top-left xywh. Invalid/degenerate boxes are dropped. Global NMS removes repeated views across overlap; it may also suppress distinct nearby players with overlapping boxes.

Pitch filtering is off: no approved shared pitch-mask artifact was available. No ground-truth-derived inference mask is used.

In [ ]:
display(pd.DataFrame([{'pre_global_nms':len(geometry['pre_nms']),'post_global_nms':len(geometry['post_nms']),'pitch_filter':manifest['pitch_filter']}]))
show('required_validation/stages/pre_nms.jpg')
show('required_validation/stages/post_nms.jpg')

## 10. Detection baseline and starting configuration
The historical-style reference uses 1280×1080 tiles, overlap 64, `imgsz=960`, reporting threshold 0.40 and retained floor 0.05. Its historical pitch mask is unavailable, so this is a documented partial reproduction. Historical project numbers are not reused as current measurements. Each candidate changes one factor relative to the 1280-input baseline where practical.

In [ ]:
grid = pd.read_csv(RUN/'development_results.csv')
reference = pd.read_csv(RUN/'reference_threshold_comparison.csv')
display(reference)
print('All rows are provisional local validation counts, not official AP.')

## 11. Validation configuration and threshold experiments
Candidate tile widths 960/1280, inference sizes 960/1280 and overlaps 64/128 are paired on the same 33 frames. The small bounded study is not an exhaustive grid. Selection maximizes local F1 at IoU 0.5, with runtime breaking ties. Global NMS is fixed; pitch on/off is blocked by the missing approved mask.

The final reporting threshold differs from the retained score floor, per-tile/global NMS IoU, evaluation IoU, and ByteTrack high/low thresholds. These control separate decisions. The reporting threshold is not applied to tracker input.

In [ ]:
display(pd.read_csv(RUN/'candidate_optima.csv')[['configuration','threshold','precision','recall','f1','fps']])
print('Selected:',manifest['selected_candidate'],manifest['selected_configuration']['operating_threshold'])
show('figures/controlled_experiments.png')
show('figures/threshold_analysis.png')

## 12. ByteTrack architecture and tracking pipeline
Top-left xywh becomes centre-based xywh before calling the installed BYTETracker. Its Kalman state tracks centre x/y, aspect ratio and height with velocities. High-score association uses IoU distance with optional score fusion and LAP assignment; remaining tracked objects associate with low-score detections using IoU-only cost (fixed second-stage limit 0.5 in this installed version). Lost tracks may reactivate; stale lost tracks are removed after the configured buffer.

New tracks require `new_track_thresh`; later arrivals require confirmation before being returned. Update is called on every consecutive frame, including empty frames. State resets per independent sequence. Sparse eval10 is rejected. Raw associated detection boxes/scores are written through returned full-input association indices. A Kalman-output mode is also tested. There is no appearance ReID or guaranteed identity through crossings.

In [ ]:
tracking = pd.DataFrame(manifest['tracking_diagnostics'])
display(tracking.drop(columns=['settings']))
show('figures/tracking_diagnostics.png')
show('figures/trajectories.png')

## 13. Detection and tracking evaluation
Local detection diagnostics use score-ordered greedy one-to-one matching at IoU 0.5, checked against the shared hand-computed fixture. TP is a matched prediction; FP is unmatched; FN is unmatched GT. Precision = TP/(TP+FP), recall = TP/(TP+FN), F1 = 2TP/(2TP+FP+FN). Object detection has no well-defined background-box TN count, so specificity, accuracy and ROC-AUC are not reported here.

Official Member 3 validators/evaluators and oracle/noisy-GT producers remain placeholders. AP/mAP/HOTA/IDF1/MOTA/official ID switches are **NOT RUN**. Track counts and lengths are local diagnostics, not substitute tracking metrics. Required full eval10 and win60s are recorded by exact coverage, including empty-prediction frames.

In [ ]:
display(pd.DataFrame(manifest['required_detection_results']))
for name,expected in [('eval10',598),('win60s',1500)]:
    coverage=json.loads((RUN/f'required_validation/{name}/coverage.json').read_text())
    assert coverage['decoded_frames']==expected
    print(name,expected,'frames; coverage verified')
print('Official E3:',manifest['official_e3'])
print('Official E4:',manifest['official_e4'])

## 14. Comparative results and runtime
Compare candidates only on the identical development frames at the same threshold or clearly stated individual optimum. Report-frame results use the selected fixed threshold. This within-match separation reduces direct frame reuse but is not an unbiased final test.

Processing time includes tile copying, Ultralytics preprocessing/inference/local NMS and global merge, with CUDA synchronization; initial warmup is included. Model load is separate. The combined cache's end-to-end timing includes sequential decoding and stage export; output CSV writing and later visualization are excluded. Framesets share a union decode pass, so subset processing FPS is measured directly, while subset-specific end-to-end FPS is not claimed.

Member 1 learns patch classification behind handcrafted proposals; YOLO jointly detects boxes and person classes using pretrained features. Member 1 uses IoU/Hungarian links; ByteTrack adds Kalman motion and low-score recovery. There is no certified paired A/B performance comparison yet.

In [ ]:
display(reference[['configuration','threshold','precision','recall','f1','fps']])
warmed = pd.read_csv(RUN/'warmed_runtime_probe.csv').groupby('configuration').processing_seconds.agg(['mean','median'])
warmed['processing_fps'] = 1/warmed['mean']
display(warmed)
print('Separate eight-frame warmed probe: setup/decode excluded. Full-run timings include cold starts and possible GPU smoke-test contention.')
display(pd.DataFrame([{'union_frames':manifest['decoded_union_frames'],'decode_included_seconds':manifest['union_decode_included_seconds'],'decode_included_fps':manifest['union_decode_included_fps']}]))

## 15. Error analysis and scientific limitations
Green boxes are local TP, red FP and blue FN at the fixed selected threshold. Inspect native-pixel crops before attributing failures. Unannotated people are a plausible source of FP, not a universal explanation. Small distant players, border fragments, score calibration and crowd overlap can cause misses or poor localization. Tracking can fragment after long gaps or change identities during crossings.

The visual examples are selected illustrations, not a prevalence estimate. In `failure_0_fp_0.jpg`, a visible touchline person holds a board and is outside the player GT: this supports the person-versus-player mismatch interpretation for that example. `failure_1000_fn_6.jpg` shows a missed white-kit player near a blue-kit player; overlap is a plausible difficulty, without proving the detector's internal cause. Predicted trajectories remain image-space paths, not calibrated speed, workload or physiological fatigue.

In [ ]:
show('figures/errors_frame_0.jpg')
show('figures/errors_frame_500.jpg')
show('figures/errors_frame_1000.jpg')
for path in sorted((RUN/'figures').glob('failure_0_*.jpg'))[:6]:
    display(Image(filename=str(path),width=220))
print('Annotated video:',RUN/'annotated_validation.mp4')
display(pd.DataFrame([json.loads((RUN/'video_verification.json').read_text())]))

## 16. Conclusions and reproducibility
The local pipeline has synthetic correctness tests and real validation evidence. Official integration remains blocked by Members 3/4; held-out M6 is prohibited. Phase 3 may fine-tune on approved train tiles, improve tiny-object recall, calibrate pitch filtering and assess appearance-based association. No Phase 3 work is executed here.

To reproduce explicitly, from the repository root with the existing environment:

```powershell
.venv/Scripts/python.exe demos/member2/scientific_workflow.py --output outputs/member2_scientific/runs/NEW_UNIQUE_RUN
.venv/Scripts/python.exe demos/member2/build_artifacts.py outputs/member2_scientific/runs/NEW_UNIQUE_RUN
.venv/Scripts/python.exe demos/member2/build_notebook.py outputs/member2_scientific/runs/NEW_UNIQUE_RUN
```

No fresh directory is overwritten. Pipeline sources, input hashes, full configurations, seeds, frame IDs, device, versions, measured timings and artifact hashes accompany the run. See Member 2 README and scientific handoff for the exact shared interfaces still required.

In [ ]:
print('Verified run:',manifest['run_id'])
print('No training, downloads or held-out testing performed by notebook execution.')
print('Verdict: PARTIAL — SHARED DEPENDENCIES BLOCKED')